In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

# 데이터 로드
master_df = pd.read_csv("data/master_dataset.csv", index_col=0, parse_dates=True)
price_df  = pd.read_csv("data/price_raw.csv",      index_col=0, parse_dates=True)

STOCK_COLS = ["삼성전자", "SK하이닉스", "NAVER", "KODEX200", "KODEX채권"]
returns_df = master_df[STOCK_COLS]

INVESTMENT = 100_000_000  # 1억 원 기준 포트폴리오
WEIGHTS    = np.array([0.25, 0.20, 0.20, 0.25, 0.10])  # 종목별 비중
CONFIDENCE = 0.95
HOLDING    = 10  # 보유기간 (일)

In [ ]:
# 1. 포트폴리오 일간 수익률
# ════════════════════════════════════════════════════════════════
portfolio_returns = (returns_df * WEIGHTS).sum(axis=1)
print(f"포트폴리오 일평균 수익률: {portfolio_returns.mean():.4f}")
print(f"포트폴리오 일간 변동성:   {portfolio_returns.std():.4f}")

포트폴리오 일평균 수익률: -0.0000
포트폴리오 일간 변동성:   0.0129


In [4]:
# 2. VaR — 히스토리컬 방식
# ════════════════════════════════════════════════════════════════
def historical_var(returns: pd.Series, confidence: float = 0.95, investment: float = 1.0) -> float:
    var_pct = np.percentile(returns, (1 - confidence) * 100)
    return abs(var_pct) * investment

hist_var_1d = historical_var(portfolio_returns, CONFIDENCE, INVESTMENT)
# N일 VaR = 1일 VaR × √N (정규분포 가정)
hist_var_nd = hist_var_1d * np.sqrt(HOLDING)

print(f"\n 히스토리컬 VaR")
print(f"  1일  VaR (95%): {hist_var_1d:,.0f}원  ({hist_var_1d/INVESTMENT*100:.2f}%)")
print(f"  {HOLDING}일 VaR (95%): {hist_var_nd:,.0f}원  ({hist_var_nd/INVESTMENT*100:.2f}%)")




 히스토리컬 VaR
  1일  VaR (95%): 1,975,037원  (1.98%)
  10일 VaR (95%): 6,245,614원  (6.25%)


In [5]:
# 3. VaR — 몬테카를로 시뮬레이션
# ════════════════════════════════════════════════════════════════
def monte_carlo_var(
    returns: pd.Series,
    confidence: float = 0.95,
    investment: float = 1.0,
    n_simulations: int = 100_000,
    holding: int = 1
) -> float:
    mu    = returns.mean()
    sigma = returns.std()
    
    # N일 보유 시 수익률 시뮬레이션
    simulated = np.random.normal(mu * holding, sigma * np.sqrt(holding), n_simulations)
    var_pct   = np.percentile(simulated, (1 - confidence) * 100)
    return abs(var_pct) * investment

np.random.seed(42)
mc_var_1d = monte_carlo_var(portfolio_returns, CONFIDENCE, INVESTMENT, holding=1)
mc_var_nd = monte_carlo_var(portfolio_returns, CONFIDENCE, INVESTMENT, holding=HOLDING)

print(f"\n 몬테카를로 VaR")
print(f"  1일  VaR (95%): {mc_var_1d:,.0f}원  ({mc_var_1d/INVESTMENT*100:.2f}%)")
print(f"  {HOLDING}일 VaR (95%): {mc_var_nd:,.0f}원  ({mc_var_nd/INVESTMENT*100:.2f}%)")




 몬테카를로 VaR
  1일  VaR (95%): 2,118,845원  (2.12%)
  10일 VaR (95%): 6,689,188원  (6.69%)


In [6]:
# 4. CVaR (Conditional VaR = Expected Shortfall)
#    VaR 초과 손실의 평균 → "최악의 경우 평균적으로 얼마?"
# ════════════════════════════════════════════════════════════════
def conditional_var(returns: pd.Series, confidence: float = 0.95, investment: float = 1.0) -> float:
    var_pct = np.percentile(returns, (1 - confidence) * 100)
    tail    = returns[returns <= var_pct]  # VaR 초과 손실 구간
    cvar    = abs(tail.mean()) * investment
    return cvar

cvar = conditional_var(portfolio_returns, CONFIDENCE, INVESTMENT)
print(f"\n CVaR / Expected Shortfall")
print(f"  CVaR (95%): {cvar:,.0f}원  ({cvar/INVESTMENT*100:.2f}%)")
print(f"  → 최악 5% 상황에서 평균 손실")


 CVaR / Expected Shortfall
  CVaR (95%): 2,614,205원  (2.61%)
  → 최악 5% 상황에서 평균 손실


In [8]:
# 5. Duration & Convexity (KODEX채권 ETF 기반)
# ════════════════════════════════════════════════════════════════
def macaulay_duration(
    face_value: float,
    coupon_rate: float,
    ytm: float,          # 만기수익률 (소수점)
    periods: int,        # 남은 이자 지급 횟수
    freq: int = 1        # 연간 이자 지급 횟수
) -> tuple[float, float, float]:

    r = ytm / freq  # 기간별 수익률
    cash_flows = [face_value * coupon_rate / freq] * periods
    cash_flows[-1] += face_value  # 마지막 기간에 원금 상환

    # 현재가치 계산
    pv_list  = [cf / (1 + r) ** (t + 1) for t, cf in enumerate(cash_flows)]
    price    = sum(pv_list)

    # Macaulay Duration = Σ(t × PV_t) / Price
    weighted = [(t + 1) / freq * pv for t, pv in enumerate(pv_list)]
    mac_dur  = sum(weighted) / price

    # Modified Duration = Macaulay / (1 + r)
    mod_dur  = mac_dur / (1 + r)

    # Convexity = Σ[t(t+1) × PV_t] / [Price × (1+r)²]
    conv_num = sum(
        (t + 1) * (t + 2) / freq**2 * pv
        for t, pv in enumerate(pv_list)
    )
    convexity = conv_num / (price * (1 + r) ** 2)

    return mac_dur, mod_dur, convexity

# 국고채 3년 현재 수익률로 계산
current_ytm   = master_df["국고채3년"].iloc[-1] / 100
mac, mod, cvx = macaulay_duration(
    face_value=10_000,
    coupon_rate=0.035,   # 표면금리 3.5%
    ytm=current_ytm,
    periods=3,           # 3년 만기
    freq=1
)

print(f"\n [Duration & Convexity] (국고채 3년 기준)")
print(f"  현재 YTM:         {current_ytm*100:.3f}%")
print(f"  Macaulay Duration: {mac:.4f}년")
print(f"  Modified Duration: {mod:.4f}")
print(f"  Convexity:         {cvx:.4f}")

# 금리 +1%p 변동 시 채권 가격 변화 예측
delta_r         = 0.01
price_change_pct = (-mod * delta_r + 0.5 * cvx * delta_r**2) * 100
print(f"\n  금리 +1%p 시 가격 변화: {price_change_pct:.4f}%")


 [Duration & Convexity] (국고채 3년 기준)
  현재 YTM:         2.597%
  Macaulay Duration: 2.9011년
  Modified Duration: 2.8276
  Convexity:         10.8995

  금리 +1%p 시 가격 변화: -2.7731%


In [9]:
# 6. NPV & IRR (사업성 평가)
# ════════════════════════════════════════════════════════════════
def calculate_npv(cash_flows: list, discount_rate: float) -> float:
    """NPV = Σ CF_t / (1+r)^t"""
    return sum(cf / (1 + discount_rate) ** t for t, cf in enumerate(cash_flows))

def calculate_irr(cash_flows: list, tol: float = 1e-6, max_iter: int = 1000) -> float:
    """이진 탐색으로 IRR 계산 (NPV=0이 되는 할인율)"""
    low, high = -0.9, 10.0
    for _ in range(max_iter):
        mid = (low + high) / 2
        npv = calculate_npv(cash_flows, mid)
        if abs(npv) < tol:
            return mid
        if npv > 0:
            low = mid
        else:
            high = mid
    return mid

# 예시: 초기 투자 1억, 향후 5년 현금흐름
sample_cf   = [-100_000_000, 25_000_000, 30_000_000, 35_000_000, 30_000_000, 25_000_000]
discount_r  = 0.08  # 할인율 8%
npv         = calculate_npv(sample_cf, discount_r)
irr         = calculate_irr(sample_cf)

print(f"\n [사업성 평가 — NPV / IRR]")
print(f"  할인율:  {discount_r*100:.1f}%")
print(f"  NPV:    {npv:,.0f}원")
print(f"  IRR:    {irr*100:.2f}%")
print(f"  판정:   {'투자 타당' if npv > 0 else '투자 부적합'}")


 [사업성 평가 — NPV / IRR]
  할인율:  8.0%
  NPV:    15,717,917원
  IRR:    13.75%
  판정:   투자 타당


In [10]:
# 7. 결과 저장
# ════════════════════════════════════════════════════════════════
results = {
    "hist_var_1d":  hist_var_1d,
    "hist_var_10d": hist_var_nd,
    "mc_var_1d":    mc_var_1d,
    "mc_var_10d":   mc_var_nd,
    "cvar_95":      cvar,
    "mac_duration": mac,
    "mod_duration": mod,
    "convexity":    cvx,
    "npv":          npv,
    "irr":          irr,
}
pd.Series(results).to_csv("data/financial_metrics.csv", header=["value"])